# 05 — Máscara agrícola Sentinel-2 60 m por campaña (2019–2025)

**Objetivo.** Generar, para cada campaña y provincia, una máscara de área agrícola estival a 60 m
independiente del INTA, y compararla con el Mapa Nacional de Cultivos del INTA y con la máscara MODIS.

**Método** (adaptado del informe técnico GP-S05-03010104-005): Random Forest (100 árboles) por provincia y
campaña sobre 6 medianas bimestrales de Sentinel-2 L1C (`COPERNICUS/S2_HARMONIZED`, escenas < 20 % de nubes;
60 variables), entrenado con 15.000 puntos por clase de una **máscara MODIS estable** (agrícola en > 14 de
las 22 campañas 2000/01–2021/22, máscaras recalculadas en el notebook `04`). Postproceso: parches de < 4
píxeles → moda en ventana de 100 m. Clasificación solo sobre los departamentos del estudio.

| Sección | Producto | Dónde | Costo GEE |
|---|---|---|---|
| 2. Funciones | etiquetas, apilado S2, `mascara_s2` | — | — |
| 3. Control Rosario 2020/21 | nueva vs máscara original del informe | interactivo (opcional) | medio |
| 4. Prueba de muestreo en La Pampa | `prueba_muestreo_lapampa_2021` | asset (tabla) | medio |
| 5. Máscaras 2019–2025 | `mascaras_s2_60m/s2_60m_AAAA_<gid>` (35 imágenes) | asset | **alto** |
| 6. Control Santa Fe 2020/21 | nueva vs original | interactivo (opcional) | medio |
| 7. Comparación con INTA y MODIS | `comparacion_mascaras_2019_2025_30m` | asset (tabla) | medio |
| 8. Métricas de la comparación | `results/tables/comparacion_mascaras_2019_2025.csv`, `metricas_mascaras_2019_2025.csv` | local | ninguno |

**Convenciones.** Campaña = año de febrero (2021 = 2020/21; año de siembra = campaña − 1). Agricultura
extensiva INTA = maíz, soja, girasol, algodón, maní, sorgo (10, 11, 12, 15, 16, 18). El INTA se lee siempre a
**30 m** (resolución nativa): sus assets tienen pirámides MEAN y a escalas más gruesas aparecen clases falsas.

**Cómo usarlo.** Se puede correr completo: las celdas que lanzan tareas omiten lo que ya existe o está en
cola, y los controles interactivos solo corren si se activan (`CORRER_CONTROLES = True`).

## 1. Configuración

In [1]:
import ee, json, collections, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", message=".*restricted mode.*")

ee.Initialize(project="ee-mnolasco")

BASE = "users/mnolasco/FORMAR_2024_2026"
COL_SALIDA = f"{BASE}/mascaras_s2_60m"
CARPETA_MODIS = f"{BASE}/mascaras_modis"                  # notebook 04
INTA = f"{BASE}/MNC_verano-{{}}"
BIMAP_ORIG = "users/mnolasco/IG-SANCOR_BiMap_60m/Bi_map_{}-{}_60m"   # máscaras del informe (hasta 2021/22)
TABLAS = "../results/tables"

CAMPANIAS = list(range(2019, 2026))
PROVINCIAS = {"Buenos Aires": 24, "Córdoba": 13, "Entre Ríos": 14, "La Pampa": 22, "Santa Fe": 4}
EXT = [10, 11, 12, 15, 16, 18]
CORRER_CONTROLES = False          # True para repetir los controles interactivos (secciones 3 y 6)

prov = ee.FeatureCollection("projects/ee-mnolasco/assets/Provincias")
deptos = ee.FeatureCollection(f"{BASE}/departments_selected")

try:
    ee.data.createAsset({"type": "IMAGE_COLLECTION"}, COL_SALIDA)
except Exception:
    pass                                                    # ya existe

/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.17) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# Utilidades para lanzar tareas sin duplicarlas
def existe(asset_id):
    try:
        ee.data.getAsset(asset_id); return True
    except Exception:
        return False

def tareas_en_cola():
    ops = ee.data.listOperations()
    return {o["metadata"].get("description") for o in ops
            if o["metadata"].get("state") in ("PENDING", "RUNNING", "READY")}

def lanzar(tarea, nombre, asset_id, en_cola):
    if existe(asset_id):
        print(f"  ya existe   {nombre}")
    elif nombre in en_cola:
        print(f"  en cola     {nombre}")
    else:
        tarea.start(); print(f"  LANZADA     {nombre}")

def estado(n=15):
    """Últimas n tareas de GEE."""
    for o in ee.data.listOperations()[:n]:
        m = o["metadata"]
        err = o.get("error", {}).get("message", "")
        print(f"{m.get('description', ''):<36} {m.get('state', ''):<10} {err}")

estado()

comparacion_mascaras_2019_2025_30m   SUCCEEDED  
diag_mascara_clases_inta_30m         SUCCEEDED  
comparacion_mascaras_2019_2025_30m   FAILED     Cannot overwrite asset 'projects/earthengine-legacy/assets/users/mnolasco/FORMAR_2024_2026/comparacion_mascaras_2019_2025'.
diag_mascara_clases_inta             SUCCEEDED  
soja_feb_otsu_2025                   SUCCEEDED  
soja_feb_otsu_2024                   SUCCEEDED  
soja_feb_otsu_2023                   SUCCEEDED  
soja_feb_otsu_2022                   SUCCEEDED  
soja_feb_otsu_2019                   SUCCEEDED  
soja_feb_otsu_2020                   SUCCEEDED  
soja_feb_otsu_2021                   SUCCEEDED  
hist_gwcci_2025                      SUCCEEDED  
hist_gwcci_2024                      SUCCEEDED  
hist_gwcci_2023                      SUCCEEDED  
hist_gwcci_2022                      SUCCEEDED  


## 2. Funciones: etiquetas, apilado Sentinel-2 y clasificación

- **Etiquetas:** máscara MODIS estable (agrícola en > 14 de 22 campañas, 2000/01–2021/22), a partir de las
  máscaras MODIS recalculadas (notebook `04`).
- **Apilado:** 6 medianas bimestrales (jun–jul, ago–sep, oct–nov, dic–ene, feb–mar, abr–may) × 10 bandas.
  `COPERNICUS/S2_HARMONIZED` corrige el desplazamiento radiométrico de la línea de base 04.00 (2022 en adelante).
- **`mascara_s2(anio, provincia, muestreo)`:** `anio` = año de siembra. `muestreo="provincia"` (método
  adoptado) toma los puntos en toda la provincia; `"deptos"` solo en los departamentos del estudio (usado
  únicamente en la prueba de la sección 4).

In [3]:
etiqueta = (ee.ImageCollection([ee.Image(f"{CARPETA_MODIS}/modis_agricola_{y}") for y in range(2001, 2023)])
            .sum().gt(14).rename("b1"))

B_S2 = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
NOMBRES = ["BLUE", "GREEN", "RED", "RED_EDGE1", "RED_EDGE2", "RED_EDGE3", "NIR", "RED_EDGE4", "SWIR1", "SWIR2"]
BIMESTRES = [(6, 7), (8, 9), (10, 11), (12, 1), (2, 3), (4, 5)]

def apilado_s2(anio, region):
    """6 medianas bimestrales (jun–may) de S2 L1C armonizado, escenas < 20 % de nubes."""
    col = (ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
           .filterDate(f"{anio}-06-01", f"{anio + 1}-06-01").filterBounds(region)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)).select(B_S2, NOMBRES))
    capas = []
    for k, (m1, m2) in enumerate(BIMESTRES):
        suf = "" if k == 0 else f"_{k}"
        capas.append(col.filter(ee.Filter.calendarRange(m1, m2, "month")).median()
                     .rename([n + suf for n in NOMBRES]))
    return ee.Image.cat(capas)

def mascara_s2(anio, nombre_prov, destino=None, muestreo="provincia"):
    """Máscara agrícola 60 m (Random Forest) de la campaña anio/anio+1."""
    gid = PROVINCIAS[nombre_prov]
    zona = prov.filter(ee.Filter.eq("gid", gid))
    deptos_prov = deptos.filter(ee.Filter.eq("provincia", nombre_prov))
    if destino is None:
        destino = deptos_prov
    region_muestreo = zona.geometry() if muestreo == "provincia" else deptos_prov.geometry()
    pila = apilado_s2(anio, zona)
    puntos = etiqueta.stratifiedSample(numPoints=15000, classBand="b1", region=region_muestreo,
                                       scale=250, seed=0, geometries=True)
    entren = pila.sampleRegions(collection=puntos, properties=["b1"], scale=60)
    rf = ee.Classifier.smileRandomForest(100).train(entren, "b1", pila.bandNames())
    clas = pila.classify(rf).clip(destino)
    parche = clas.connectedPixelCount(15, False)
    moda = clas.focal_mode(radius=100, kernelType="square", units="meters")
    return (clas.where(parche.lt(4), moda).rename("b1").toByte()
            .set({"campania": anio + 1, "provincia": nombre_prov, "gid": gid, "muestreo": muestreo}))

## 3. Control: Rosario 2020/21 (opcional, interactivo)

Compara la máscara generada con este código con la del informe técnico en el departamento Rosario.
Resultado registrado (con las etiquetas MODIS originales): 101.207 vs 101.179 ha (+0,0 %), coincidencia
96,4 % de la máscara existente. Con las etiquetas recalculadas el resultado puede variar levemente.

In [4]:
if CORRER_CONTROLES:
    ros = deptos.filter(ee.Filter.eq("link", "82084"))
    nueva = mascara_s2(2020, "Santa Fe", destino=ros).eq(1)
    vieja = ee.Image(BIMAP_ORIG.format(2020, 2021)).eq(1).clip(ros)
    def ha(m):
        return (m.multiply(ee.Image.pixelArea()).divide(1e4)
                .reduceRegion(ee.Reducer.sum(), ros.geometry(), scale=60, crs="EPSG:32720",
                              maxPixels=1e13, tileScale=8).values().get(0))
    a_n, a_v, a_c = ee.List([ha(nueva), ha(vieja), ha(nueva.And(vieja))]).getInfo()
    print(f"Rosario 2020/21 — nueva: {a_n:,.0f} ha | existente: {a_v:,.0f} ha | "
          f"diferencia {100 * (a_n / a_v - 1):+.1f} % | coincidencia {100 * a_c / a_v:.1f} % de la existente")
else:
    print("Control omitido (CORRER_CONTROLES = False)")

Control omitido (CORRER_CONTROLES = False)


## 4. Prueba de muestreo en La Pampa (campaña 2020/21)

La máscara del informe sobreestima el área agrícola en La Pampa. Se compararon tres variantes en los 5
departamentos pampeanos: máscara original, muestreo provincial con etiquetas recalculadas y muestreo
restringido a los departamentos. Resultado: el muestreo restringido corrige la sobreestimación pero omite
el 27 % de la soja (63 % en Trenel), porque en zonas de expansión agrícola reciente la máscara estable
etiqueta como no agrícolas lotes hoy agrícolas. **Decisión: muestreo provincial en todas las provincias.**

In [5]:
nombre = "prueba_muestreo_lapampa_2021"
asset = f"{BASE}/{nombre}"

lp = deptos.filter(ee.Filter.eq("provincia", "La Pampa")).select(["link"])
inta21 = ee.Image(INTA.format(2021))
ext21 = inta21.remap(EXT, [1] * len(EXT), 0).unmask(0)
soja21 = inta21.eq(11).unmask(0)
versiones = {"original": ee.Image(BIMAP_ORIG.format(2020, 2021)).eq(1).unmask(0),
             "provincia": mascara_s2(2020, "La Pampa", muestreo="provincia").eq(1).unmask(0),
             "deptos": mascara_s2(2020, "La Pampa", muestreo="deptos").eq(1).unmask(0)}
bandas = [ext21.rename("ext"), soja21.rename("soja")]
for k, M in versiones.items():
    bandas += [M.rename(k), ext21.And(M).rename(f"{k}_ext"), soja21.And(M.Not()).rename(f"{k}_sojafuera")]
res_lp = (ee.Image.cat(bandas).multiply(ee.Image.pixelArea().divide(1e4))
          .reduceRegions(lp, ee.Reducer.sum(), scale=30, crs="EPSG:32720", tileScale=8)
          .map(lambda f: f.setGeometry(f.geometry().centroid(100))))
lanzar(ee.batch.Export.table.toAsset(res_lp, nombre, asset), nombre, asset, tareas_en_cola())

  ya existe   prueba_muestreo_lapampa_2021


In [6]:
# Resultados de la prueba (si la tabla existe)
if existe(f"{BASE}/prueba_muestreo_lapampa_2021"):
    feats = ee.FeatureCollection(f"{BASE}/prueba_muestreo_lapampa_2021").getInfo()["features"]
    info = pd.read_csv(f"{TABLAS}/seleccion_imagen_2021.csv", dtype={"link": str})[["link", "departamento"]]
    pl = pd.DataFrame([f["properties"] for f in feats])
    pl["link"] = pl["link"].astype(str).str.zfill(5)
    pl = pl.merge(info, on="link", how="left")

    VERS = ["original", "provincia", "deptos"]
    def ind(s, k):
        return {"soja_fuera_%": 100 * s[f"{k}_sojafuera"] / s["soja"],
                "mascara_no_agri_%": 100 * (1 - s[f"{k}_ext"] / s[k]),
                "area_vs_INTA_%": 100 * s[k] / s["ext"]}
    tot = pl.sum(numeric_only=True)
    print("La Pampa (5 departamentos), campaña 2020/21 — total:")
    print(pd.DataFrame({k: ind(tot, k) for k in VERS}).T.round(1).to_string())
    print("\nÁrea de la máscara respecto del INTA (%), por departamento:")
    print(pd.DataFrame({k: pl.apply(lambda r: 100 * r[k] / r["ext"], axis=1) for k in VERS})
          .set_index(pl["departamento"]).round(1).to_string())
    print("\nSoja fuera de la máscara (%), por departamento:")
    print(pd.DataFrame({k: pl.apply(lambda r: 100 * r[f"{k}_sojafuera"] / r["soja"], axis=1) for k in VERS})
          .set_index(pl["departamento"]).round(1).to_string())

La Pampa (5 departamentos), campaña 2020/21 — total:
           soja_fuera_%  mascara_no_agri_%  area_vs_INTA_%
original            8.2               30.2           130.0
provincia           7.3               27.9           127.4
deptos             26.9                6.4            72.2

Área de la máscara respecto del INTA (%), por departamento:
              original  provincia  deptos
departamento                             
Trenel           115.1      117.2    34.9
Chapaleufú       138.2      132.7    94.9
Maracó           137.7      133.0    90.2
Quemú Quemú      124.9      124.2    64.6
Realicó          125.4      124.2    53.8

Soja fuera de la máscara (%), por departamento:
              original  provincia  deptos
departamento                             
Trenel            24.8       21.9    63.1
Chapaleufú         0.8        0.9     7.8
Maracó             2.2        2.2    13.4
Quemú Quemú       12.7       11.2    38.8
Realicó           12.4       10.7    40.0


## 5. Máscaras 2019–2025 (35 imágenes)

Una imagen por provincia y campaña en la colección `mascaras_s2_60m`, nombrada `s2_60m_<campaña>_<gid>`, con
propiedades `campania`, `provincia`, `gid` y `muestreo`. Pirámides con política MODE (dato categórico).
Mosaico de una campaña: `ee.ImageCollection(COL_SALIDA).filter(ee.Filter.eq("campania", c)).mosaic()`.

In [7]:
en_cola = tareas_en_cola()
for c in CAMPANIAS:
    for p, gid in PROVINCIAS.items():
        destino = deptos.filter(ee.Filter.eq("provincia", p))
        nombre = f"s2_60m_{c}_{gid}"
        asset = f"{COL_SALIDA}/{nombre}"
        t = ee.batch.Export.image.toAsset(
            image=mascara_s2(c - 1, p, muestreo="provincia"), description=nombre, assetId=asset,
            region=destino.geometry().bounds(), scale=60, crs="EPSG:32720",
            pyramidingPolicy={"b1": "mode"}, maxPixels=1e13)
        lanzar(t, nombre, asset, en_cola)

  ya existe   s2_60m_2019_24
  ya existe   s2_60m_2019_13
  ya existe   s2_60m_2019_14
  ya existe   s2_60m_2019_22
  ya existe   s2_60m_2019_4
  ya existe   s2_60m_2020_24
  ya existe   s2_60m_2020_13
  ya existe   s2_60m_2020_14
  ya existe   s2_60m_2020_22
  ya existe   s2_60m_2020_4
  ya existe   s2_60m_2021_24
  ya existe   s2_60m_2021_13
  ya existe   s2_60m_2021_14
  ya existe   s2_60m_2021_22
  ya existe   s2_60m_2021_4
  ya existe   s2_60m_2022_24
  ya existe   s2_60m_2022_13
  ya existe   s2_60m_2022_14
  ya existe   s2_60m_2022_22
  ya existe   s2_60m_2022_4
  ya existe   s2_60m_2023_24
  ya existe   s2_60m_2023_13
  ya existe   s2_60m_2023_14
  ya existe   s2_60m_2023_22
  ya existe   s2_60m_2023_4
  ya existe   s2_60m_2024_24
  ya existe   s2_60m_2024_13
  ya existe   s2_60m_2024_14
  ya existe   s2_60m_2024_22
  ya existe   s2_60m_2024_4
  ya existe   s2_60m_2025_24
  ya existe   s2_60m_2025_13
  ya existe   s2_60m_2025_14
  ya existe   s2_60m_2025_22
  ya existe   s2_60m

In [8]:
# Inventario de la colección
ids = sorted(a["name"].split("/")[-1] for a in ee.data.listAssets({"parent": COL_SALIDA})["assets"])
esperados = {f"s2_60m_{c}_{g}" for c in CAMPANIAS for g in PROVINCIAS.values()}
print(f"Assets: {len(ids)} de {len(esperados)} | Faltan: {sorted(esperados - set(ids)) or 'ninguno'}")

Assets: 35 de 35 | Faltan: ninguno


## 6. Control: Santa Fe 2020/21 (opcional, interactivo)

Máscara nueva vs máscara del informe en los departamentos de Santa Fe. Resultado registrado: superficie
−2,1 % (3.099.095 vs 3.166.839 ha); intersección/unión 92,7 %.

In [9]:
if CORRER_CONTROLES:
    nueva = ee.Image(f"{COL_SALIDA}/s2_60m_2021_4").eq(1)
    orig = ee.Image(BIMAP_ORIG.format(2020, 2021)).eq(1)
    region_sf = deptos.filter(ee.Filter.eq("provincia", "Santa Fe")).geometry()
    print(ee.Image(f"{COL_SALIDA}/s2_60m_2021_4").toDictionary().getInfo())
    areas = (ee.Image.cat(nueva.rename("nueva"), orig.rename("orig"),
                          nueva.And(orig).rename("ambas"), nueva.Or(orig).rename("alguna"))
             .multiply(ee.Image.pixelArea()).divide(1e4)
             .reduceRegion(ee.Reducer.sum(), region_sf, 60, "EPSG:32720", maxPixels=1e13, tileScale=4)
             .getInfo())
    print({k: round(v) for k, v in areas.items()})
    print(f"Diferencia de área: {100 * (areas['nueva'] / areas['orig'] - 1):+.1f} %")
    print(f"Coincidencia (ambas/alguna): {100 * areas['ambas'] / areas['alguna']:.1f} %")
else:
    print("Control omitido (CORRER_CONTROLES = False)")

Control omitido (CORRER_CONTROLES = False)


## 7. Comparación de las máscaras con el INTA (asset)

Por departamento y campaña, superficie (ha) de: agricultura extensiva y soja del INTA; y para cada máscara
(S2 nueva, MODIS de la misma campaña y, hasta 2021/22, la S2 original del informe) su área total, la soja
INTA que contiene y la superficie no agrícola extensiva que incluye. **Se calcula a 30 m** (resolución
nativa del INTA; ver nota sobre pirámides en la introducción).

*Nota:* el valor 255 del INTA (sin dato, 0–0,7 % de la máscara) no se excluyó de la máscara de validez y
cuenta como superficie no agrícola (efecto < 1 pp). En una versión futura: `valido = valido.And(inta.neq(255))`.

In [10]:
def bandas_campania(c):
    inta = ee.Image(INTA.format(c)).select(0)
    valido = inta.mask().reduce(ee.Reducer.min()).gt(0)
    soja = inta.eq(11)
    ag = inta.remap(EXT, [1] * len(EXT), 0)
    noag = valido.And(ag.Not())
    mascaras = {
        "s2": ee.ImageCollection(COL_SALIDA).filter(ee.Filter.eq("campania", c)).mosaic().eq(1),
        "modis": ee.Image(f"{CARPETA_MODIS}/modis_agricola_{c}").select(0).eq(1),
    }
    if c <= 2022:
        mascaras["orig"] = ee.Image(BIMAP_ORIG.format(c - 1, c)).eq(1)
    b = [ag.rename("inta_ag"), soja.rename("inta_soja")]
    for k, m in mascaras.items():
        m = m.unmask(0)
        b += [m.rename(f"{k}_area"), m.And(soja).rename(f"{k}_soja"), m.And(noag).rename(f"{k}_noag")]
    return ee.Image.cat(b).updateMask(valido).multiply(ee.Image.pixelArea()).divide(1e4)   # ha

def tabla_campania(c):
    return (bandas_campania(c)
            .reduceRegions(deptos, ee.Reducer.sum(), 30, "EPSG:32720", tileScale=8)
            .map(lambda f: f.set("campania", c).setGeometry(f.geometry().centroid(100))))

nombre = "comparacion_mascaras_2019_2025_30m"
asset = f"{BASE}/{nombre}"
fc = ee.FeatureCollection([tabla_campania(c) for c in CAMPANIAS]).flatten()
lanzar(ee.batch.Export.table.toAsset(fc, nombre, asset), nombre, asset, tareas_en_cola())

  ya existe   comparacion_mascaras_2019_2025_30m


## 8. Métricas de la comparación (local)

Tres indicadores por máscara: (i) **soja fuera**: fracción de la soja INTA que la máscara omite (error
irrecuperable en las etapas siguientes); (ii) **máscara no agrícola**: fracción de la máscara que el INTA no
clasifica como agricultura extensiva; (iii) **área / INTA**: superficie de la máscara respecto de la
agricultura extensiva del INTA. El CSV por departamento lo usa el notebook `06` (superficie de soja).

In [11]:
asset = f"{BASE}/comparacion_mascaras_2019_2025_30m"
cmp = ee.data.computeFeatures({"expression": ee.FeatureCollection(asset), "fileFormat": "PANDAS_DATAFRAME"})
cmp = cmp.drop(columns=["geo"], errors="ignore")
cmp.to_csv(f"{TABLAS}/comparacion_mascaras_2019_2025.csv", index=False)
print(cmp.shape, "(esperado: 665 filas = 95 deptos × 7 campañas)")

def metricas(g, m):
    s = g.sum(min_count=1)
    return pd.Series({"soja_fuera_%": 100 * (1 - s[f"{m}_soja"] / s["inta_soja"]),
                      "mascara_no_agri_%": 100 * s[f"{m}_noag"] / s[f"{m}_area"],
                      "area_vs_INTA_%": 100 * s[f"{m}_area"] / s["inta_ag"]})

def tabla(por):
    out = []
    for m in ["s2", "modis", "orig"]:
        sub = cmp.dropna(subset=[f"{m}_area"])
        t = sub.groupby(por).apply(lambda g: metricas(g, m)).round(1)
        t["mascara"] = m
        out.append(t.reset_index())
    return pd.concat(out)

pd.set_option("display.width", 150)
por_campania = tabla(["campania"])
por_provincia = tabla(["provincia"])
pd.concat([por_campania.assign(nivel="campania"), por_provincia.assign(nivel="provincia")]) \
  .to_csv(f"{TABLAS}/metricas_mascaras_2019_2025.csv", index=False)

print("\n== Por campaña ==")
print(por_campania.pivot(index="campania", columns="mascara").to_string())
print("\n== Campañas 2020–2022 (las tres máscaras disponibles) ==")
c22 = cmp[cmp.campania.between(2020, 2022)]
for m in ["s2", "modis", "orig"]:
    print(f"{m:<6}", metricas(c22, m).round(1).to_dict())
print("\n== Por provincia (todas las campañas) ==")
print(por_provincia.pivot(index="provincia", columns="mascara").to_string())

(665, 25) (esperado: 665 filas = 95 deptos × 7 campañas)

== Por campaña ==
         soja_fuera_%             mascara_no_agri_%             area_vs_INTA_%              
mascara         modis  orig    s2             modis  orig    s2          modis   orig     s2
campania                                                                                    
2019             30.7   8.3   9.1              22.8  22.5  21.7           85.6  113.6  111.4
2020             28.8   8.9   9.8              13.9  10.7  10.0           82.6   99.3   97.4
2021             25.2  10.5  10.9              11.7   7.7   7.1           83.1   93.6   92.2
2022             26.1   7.1   7.8              14.0  11.2  10.3           84.3   99.1   97.4
2023             36.8   NaN   9.6              23.0   NaN  18.2           83.8    NaN  110.0
2024             25.1   NaN   9.2              14.2   NaN   9.3           86.5    NaN   95.7
2025             32.4   NaN   8.6              12.3   NaN  10.0           76.8    NaN  

## Registro de resultados

- Control de implementación: Rosario 2020/21 +0,0 % (96,4 % de coincidencia); Santa Fe 2020/21 −2,1 %
  (intersección/unión 92,7 %).
- La Pampa: el muestreo restringido a los departamentos sobrecorrige (27 % de soja omitida); se adopta el
  muestreo provincial y la sobreestimación queda como limitación (28–41 % de la máscara es "No agrícola" del
  INTA; ver `manuscript/resultados_evaluacion_2019_2025.md`).
- **Corrección (24-oct-2026):** la primera comparación 2019–2025 se calculó a 60 m y estaba sesgada por las
  pirámides MEAN del INTA (clases falsas; soja subestimada 5–8 %). Se recalculó a 30 m
  (`comparacion_mascaras_2019_2025_30m`); las métricas de la sección 8 corresponden a esa versión.